# Character-level SampleGPT

In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
%pip install numpy matplotlib

import math
from pathlib import Path

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn import functional as F

# Reproducibility
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

# Model / training settings
batch_size = 64
block_size = 256
max_iters = 15000          # best checkpoint is saved along the way, so overshooting is safe
warmup_iters = 300
eval_interval = 500
eval_iters = 100
learning_rate = 3e-4
min_lr = 3e-5              # cosine decays down to this
grad_clip = 1.0

n_embd = 96
n_head = 4
n_layer = 4
dropout = 0.2

checkpoint_path = "lunar_gpt_best.pth"

# bfloat16 autocast on CUDA, no-op on CPU
use_amp = (device == "cuda") and torch.cuda.is_bf16_supported()
amp_ctx = (
    torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if use_amp
    else torch.autocast(device_type="cpu", enabled=False)
)

print("Device:", device, "| bf16 autocast:", use_amp)

In [ ]:
data_path = Path("lunar_samples.txt")

if not data_path.exists():
    raise FileNotFoundError(
        f"Corpus file not found: {data_path.resolve()}"
    )

text = data_path.read_text(encoding="utf-8")

print("Dataset length:", len(text))
print()
print(text[:500])

In [ ]:
# Character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    # Skip characters that never appeared in the corpus
    # (a stray curly quote in a prompt would otherwise KeyError)
    return [stoi[c] for c in s if c in stoi]

def decode(tokens):
    return "".join(itos[i] for i in tokens)

print("Vocabulary size:", vocab_size)

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)

# 90% training, 10% validation
n = int(0.9 * len(data))

# The whole dataset easily fits in memory, so park it on the
# device once — get_batch then does no host->device copies.
train_data = data[:n].to(device)
val_data = data[n:].to(device)

print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))

In [ ]:
def get_batch(split):
    src = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(len(src) - block_size, (batch_size,), device=device)

    # Input sequences and targets shifted one character forward
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + block_size + 1] for i in ix])

    return x, y

In [ ]:
class CausalSelfAttention(nn.Module):
    """All attention heads computed in one fused operation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        # One projection producing Q, K, V for every head at once
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        # [B, T, C] -> [B, n_head, T, head_size]
        q = q.view(B, T, self.n_head, -1).transpose(1, 2)
        k = k.view(B, T, self.n_head, -1).transpose(1, 2)
        v = v.view(B, T, self.n_head, -1).transpose(1, 2)

        # Causal mask, scaling, softmax, and attention dropout
        # all happen inside this one call.
        out = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=dropout if self.training else 0.0,
        )

        # [B, n_head, T, head_size] -> [B, T, C]
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Feed-forward part of transformer."""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """One transformer block (pre-norm)."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # attention + residual
        x = x + self.ffwd(self.ln2(x))  # feed-forward + residual
        return x


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying: output head shares the token embedding matrix
        self.lm_head.weight = self.token_embedding.weight

        # GPT-2-style init
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            # Residual projections get a smaller init so the residual
            # stream doesn't blow up as depth increases
            if name.endswith("proj.weight") or name.endswith("net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)                       # character meaning
        pos_emb = self.position_embedding(
            torch.arange(T, device=idx.device)
        )                                                          # character position

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        # Raw prediction scores: [batch, sequence, vocab]
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=20):
        was_training = self.training
        self.eval()  # disable dropout during generation

        for _ in range(max_new_tokens):

            # Model can only use block_size characters of context
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)

            # Keep only the prediction at the final position:
            # [B, T, vocab] -> [B, vocab]
            logits = logits[:, -1, :]

            if temperature == 0:
                # Greedy decoding
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                # Lower temperature -> sharper, higher -> more random
                logits = logits / temperature

                if top_k is not None:
                    # Keep only the k largest logits
                    values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    cutoff = values[:, [-1]]
                    logits = logits.masked_fill(logits < cutoff, float("-inf"))

                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        if was_training:
            self.train()

        return idx

In [ ]:
model = GPTLanguageModel().to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_params / 1e6:.3f} million")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Linear warmup, then cosine decay down to min_lr
def lr_lambda(step):
    if step < warmup_iters:
        return (step + 1) / warmup_iters
    progress = (step - warmup_iters) / max(1, max_iters - warmup_iters)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return (min_lr / learning_rate) + (1 - min_lr / learning_rate) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Optional: big speedup on Linux/WSL with a recent PyTorch.
# Comment out if it causes problems on your setup (e.g. native Windows).
# model = torch.compile(model)

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split)
            with amp_ctx:
                _, loss = model(x, y)
            losses.append(loss.item())
        results[split] = sum(losses) / len(losses)
    model.train()
    return results

In [ ]:
steps = []
train_losses = []
val_losses = []
best_val = float("inf")

model.train()

for step in range(max_iters):

    # Evaluate occasionally
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()

        steps.append(step)
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])

        marker = ""
        if losses["val"] < best_val:
            best_val = losses["val"]
            torch.save({
                "model": model.state_dict(),
                "chars": chars,
                "config": dict(
                    n_embd=n_embd, n_head=n_head, n_layer=n_layer,
                    block_size=block_size, vocab_size=vocab_size,
                    dropout=dropout,
                ),
                "step": step,
                "val_loss": best_val,
            }, checkpoint_path)
            marker = "  <- saved"

        print(
            f"step {step:5d} | "
            f"train {losses['train']:.4f} | "
            f"val {losses['val']:.4f} | "
            f"lr {scheduler.get_last_lr()[0]:.2e}"
            f"{marker}"
        )

    # Training step
    xb, yb = get_batch("train")

    with amp_ctx:
        _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    scheduler.step()

print()
print(f"Best validation loss: {best_val:.4f} (saved to {checkpoint_path})")

In [ ]:
plt.plot(steps, train_losses, label="Training")
plt.plot(steps, val_losses, label="Validation")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# Reload the best checkpoint before generating
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model"])
print(f"Loaded checkpoint from step {ckpt['step']} (val loss {ckpt['val_loss']:.4f})")

prompt = """LUNAR SAMPLE 10017

INTRODUCTION

"""

context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=1000,
    temperature=0.7,
    top_k=15,
)

print(decode(generated[0].tolist()))